20260215 NASDAQ 전체 + S&P500 종목 + 주요 ETF 500개를 합쳐서 중복을 제거한 리스트를 만듭니다.

In [2]:
# 01_download_us.py
import FinanceDataReader as fdr
import pandas as pd
import os
from tqdm import tqdm
from datetime import datetime, timedelta
import warnings

# 경고 메시지 무시 (권장)
warnings.filterwarnings('ignore')

# --- 설정 ---
MARKET_NAME = 'US_ALL'  # 저장될 폴더명
START_DATE = (datetime.now() - timedelta(days=730)).strftime('%Y-%m-%d')
SAVE_DIR = f"./Raw_Data/{MARKET_NAME}"
MIN_TRADING_VALUE_USD = 5_000_000  # 최소 거래대금 (500만 달러)

# 폴더 생성
os.makedirs(SAVE_DIR, exist_ok=True)

def get_merged_stock_list():
    print("📋 종목 리스트 수집 및 병합 중...")
    
    # 1. NASDAQ 전체
    print("   - NASDAQ 리스트 가져오는 중...")
    nasdaq = fdr.StockListing('NASDAQ')
    nasdaq['Type'] = 'NASDAQ'
    
    # 2. S&P 500 (NYSE 포함 주요 종목)
    print("   - S&P 500 리스트 가져오는 중...")
    sp500 = fdr.StockListing('S&P500')
    sp500['Type'] = 'S&P500'
    
    # 3. 미국 ETF (상위 500개)
    print("   - US ETF 리스트 가져오는 중...")
    etf = fdr.StockListing('ETF/US')
    etf['Type'] = 'ETF'
    # 보통 ETF 리스트는 순서가 섞여있을 수 있으니, 상위 500개를 끊습니다.
    # (더 정확히 하려면 나중에 거래대금 필터로 걸러집니다)
    etf = etf.head(500) 

    # 4. 데이터 병합 (Symbol, Name 컬럼만 추출하여 병합)
    # 각 데이터프레임의 컬럼명이 다를 수 있어 통일 작업
    cols = ['Symbol', 'Name', 'Type']
    
    df_list = []
    for df in [nasdaq, sp500, etf]:
        # 필요한 컬럼만 선택 (없는 컬럼은 에러 방지)
        available_cols = [c for c in cols if c in df.columns]
        temp_df = df[available_cols].copy()
        df_list.append(temp_df)
        
    merged_df = pd.concat(df_list, ignore_index=True)
    
    # 5. 중복 제거 (티커 기준)
    initial_len = len(merged_df)
    merged_df.drop_duplicates(subset=['Symbol'], inplace=True)
    print(f"✅ 리스트 병합 완료: 총 {initial_len} -> {len(merged_df)}개 (중복 제거됨)")
    
    return merged_df

def download_us_data():
    stocks = get_merged_stock_list()
    
    print(f"🚀 총 {len(stocks)}개 미국 종목 다운로드 시작...")
    
    success = 0
    fail_count = 0
    
    for idx, row in tqdm(stocks.iterrows(), total=len(stocks)):
        code = row['Symbol']
        name = row['Name']
        
        try:
            # 1. 데이터 다운로드
            df = fdr.DataReader(code, start=START_DATE)
            
            if df.empty or len(df) < 100: 
                continue 
            
            # 2. 거래대금 필터링
            close_price = pd.to_numeric(df['Close'], errors='coerce')
            volume = pd.to_numeric(df['Volume'], errors='coerce')
            
            # 최근 20일 평균 거래대금 계산
            avg_val = (close_price * volume).rolling(20).mean().iloc[-1]
            
            if pd.isna(avg_val): continue

            # 최소 거래대금 미만 스킵
            if avg_val < MIN_TRADING_VALUE_USD:
                continue 
            
            # 3. 파일 저장
            safe_name = str(name).replace('/', '_').replace(':', '_').replace('"', '').replace('*', '').replace('?', '')
            file_path = f"{SAVE_DIR}/{code}_{safe_name}.csv"
            
            df.to_csv(file_path, index=True)
            success += 1
            
        except Exception as e:
            fail_count += 1
            if fail_count <= 5:
                print(f"\n❌ Error [{name}({code})]: {e}")
                if "No module named 'yfinance'" in str(e):
                    print("\n🚨 [필독] yfinance 라이브러리가 없습니다! 'pip install yfinance'를 실행하세요.")
                    break
            continue

    print(f"\n✅ 다운로드 완료! 성공: {success}개")
    print(f"📁 저장 위치: {SAVE_DIR}")

if __name__ == "__main__":
    download_us_data()

📋 종목 리스트 수집 및 병합 중...
   - NASDAQ 리스트 가져오는 중...


100%|██████████| 3827/3827 [00:05<00:00, 668.83it/s]


   - S&P 500 리스트 가져오는 중...
   - US ETF 리스트 가져오는 중...


0it [00:00, ?it/s]


ValueError: No objects to concatenate

KOSPI, KOSDAQ 전 종목 + 한국 ETF 전 종목을 병합합니다.

한국장은 티커 컬럼명이 Code이므로 이를 기준으로 통합합니다.

거래대금 기준은 원화(KRW) 65억 원으로 설정했습니다

In [2]:
# 01_download_kr.py
import FinanceDataReader as fdr
import pandas as pd
import os
from tqdm import tqdm
from datetime import datetime, timedelta
import warnings
import time     # 추가: 딜레이를 주기 위함
import random   # 추가: 랜덤한 시간으로 쉬기 위함

warnings.filterwarnings('ignore')

# --- 설정 ---
MARKET_NAME = 'KR_ALL' # 저장될 폴더명
START_DATE = (datetime.now() - timedelta(days=730)).strftime('%Y-%m-%d')
SAVE_DIR = f"./Raw_Data/{MARKET_NAME}"
MIN_TRADING_VALUE_KRW = 6_500_000_000  # 약 65억 원

# 폴더 생성
os.makedirs(SAVE_DIR, exist_ok=True)

def get_kr_merged_list():
    print("📋 한국 종목 리스트 수집 및 병합 중...")
    
    # 1. KOSPI
    print("   - KOSPI 리스트 가져오는 중...")
    kospi = fdr.StockListing('KOSPI')
    
    # 2. KOSDAQ
    print("   - KOSDAQ 리스트 가져오는 중...")
    kosdaq = fdr.StockListing('KOSDAQ')
    
    # 3. 한국 ETF
    print("   - 한국 ETF 리스트 가져오는 중...")
    etf = fdr.StockListing('ETF/KR')
    
    # 4. 병합 (Code, Name 컬럼 통일)
    for df in [kospi, kosdaq, etf]:
        if 'Symbol' in df.columns:
            df.rename(columns={'Symbol': 'Code'}, inplace=True)

    cols = ['Code', 'Name']
    df_list = [kospi[cols], kosdaq[cols], etf[cols]]
    
    merged_df = pd.concat(df_list, ignore_index=True)
    
    # 5. 중복 제거
    initial_len = len(merged_df)
    merged_df.drop_duplicates(subset=['Code'], inplace=True)
    print(f"✅ 리스트 병합 완료: 총 {initial_len} -> {len(merged_df)}개 (중복 제거됨)")
    
    return merged_df

def download_kr_data():
    stocks = get_kr_merged_list()
    
    print(f"🚀 총 {len(stocks)}개 한국 종목 다운로드 시작...")
    
    success = 0
    fail_count = 0
    skip_count = 0  # 이미 다운로드된 종목 카운트
    
    for idx, row in tqdm(stocks.iterrows(), total=len(stocks)):
        code = row['Code'] 
        name = row['Name']
        
        # 💡 [핵심 추가 1] 파일 경로를 먼저 만들고 이미 파일이 있는지 검사 (이어받기 기능)
        safe_name = str(name).replace('/', '_').replace(':', '_').replace('"', '').replace('*', '')
        file_path = f"{SAVE_DIR}/{code}_{safe_name}.csv"
        
        if os.path.exists(file_path):
            skip_count += 1
            continue  # 이미 다운로드 된 파일이면 다음 종목으로 넘어감
        
        try:
            # 💡 [핵심 추가 2] 서버 차단 방지를 위한 랜덤 휴식 (0.1 ~ 0.5초 사이)
            time.sleep(random.uniform(0.1, 0.2))
            
            # 1. 데이터 다운로드
            df = fdr.DataReader(code, start=START_DATE)
            
            if df.empty or len(df) < 100: 
                continue 
            
            # 2. 거래대금 필터링
            close_price = pd.to_numeric(df['Close'], errors='coerce')
            volume = pd.to_numeric(df['Volume'], errors='coerce')
            
            # 최근 20일 평균 거래대금
            avg_val = (close_price * volume).rolling(20).mean().iloc[-1]
            
            if pd.isna(avg_val): continue

            # 최소 거래대금 미만 스킵
            if avg_val < MIN_TRADING_VALUE_KRW:
                continue 
            
                        # 3. 파일 저장
            df.to_csv(file_path, index=True)
            success += 1
            
        except Exception as e:
            fail_count += 1
            if fail_count <= 5:
                print(f"\n❌ Error [{name}({code})]: {e}")
            continue

    print(f"\n✅ 다운로드 완료! 신규 저장: {success}개 (이미 존재하여 건너뜀: {skip_count}개)")
    print(f"📁 저장 위치: {SAVE_DIR}")

if __name__ == "__main__":
    download_kr_data()

📋 한국 종목 리스트 수집 및 병합 중...
   - KOSPI 리스트 가져오는 중...
   - KOSDAQ 리스트 가져오는 중...
   - 한국 ETF 리스트 가져오는 중...
✅ 리스트 병합 완료: 총 3839 -> 3839개 (중복 제거됨)
🚀 총 3839개 한국 종목 다운로드 시작...


  7%|▋         | 269/3839 [01:02<13:42,  4.34it/s]

: 

In [ ]:
# 01_download_kr.py
import FinanceDataReader as fdr
import pandas as pd
import os
from tqdm import tqdm
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# --- 설정 ---
MARKET_NAME = 'KR_ALL' # 저장될 폴더명
START_DATE = (datetime.now() - timedelta(days=730)).strftime('%Y-%m-%d')
SAVE_DIR = f"./Raw_Data/{MARKET_NAME}"
MIN_TRADING_VALUE_KRW = 6_500_000_000  # 약 65억 원

# 폴더 생성
os.makedirs(SAVE_DIR, exist_ok=True)

def get_kr_merged_list():
    print("📋 한국 종목 리스트 수집 및 병합 중...")
    
    # 1. KOSPI
    print("   - KOSPI 리스트 가져오는 중...")
    kospi = fdr.StockListing('KOSPI')
    
    # 2. KOSDAQ
    print("   - KOSDAQ 리스트 가져오는 중...")
    kosdaq = fdr.StockListing('KOSDAQ')
    
    # 3. 한국 ETF
    print("   - 한국 ETF 리스트 가져오는 중...")
    etf = fdr.StockListing('ETF/KR')
    
    # 4. 병합 (Code, Name 컬럼 통일)
    # fdr.StockListing 결과에서 Symbol은 'Code' 또는 'Symbol'로 나올 수 있음
    for df in [kospi, kosdaq, etf]:
        if 'Symbol' in df.columns:
            df.rename(columns={'Symbol': 'Code'}, inplace=True)

    cols = ['Code', 'Name']
    df_list = [kospi[cols], kosdaq[cols], etf[cols]]
    
    merged_df = pd.concat(df_list, ignore_index=True)
    
    # 5. 중복 제거
    initial_len = len(merged_df)
    merged_df.drop_duplicates(subset=['Code'], inplace=True)
    print(f"✅ 리스트 병합 완료: 총 {initial_len} -> {len(merged_df)}개 (중복 제거됨)")
    
    return merged_df

def download_kr_data():
    stocks = get_kr_merged_list()
    
    print(f"🚀 총 {len(stocks)}개 한국 종목 다운로드 시작...")
    
    success = 0
    fail_count = 0
    
    for idx, row in tqdm(stocks.iterrows(), total=len(stocks)):
        code = row['Code'] # 한국장은 Code
        name = row['Name']
        
        try:
            # 1. 데이터 다운로드
            df = fdr.DataReader(code, start=START_DATE)
            
            if df.empty or len(df) < 100: 
                continue 
            
            # 2. 거래대금 필터링
            close_price = pd.to_numeric(df['Close'], errors='coerce')
            volume = pd.to_numeric(df['Volume'], errors='coerce')
            
            # 최근 20일 평균 거래대금
            avg_val = (close_price * volume).rolling(20).mean().iloc[-1]
            
            if pd.isna(avg_val): continue

            # 최소 거래대금 미만 스킵
            if avg_val < MIN_TRADING_VALUE_KRW:
                continue 
            
            # 3. 파일 저장
            # 파일명에 특수문자 제거
            safe_name = str(name).replace('/', '_').replace(':', '_').replace('"', '').replace('*', '')
            file_path = f"{SAVE_DIR}/{code}_{safe_name}.csv"
            
            df.to_csv(file_path, index=True)
            success += 1
            
        except Exception as e:
            fail_count += 1
            if fail_count <= 5:
                print(f"\n❌ Error [{name}({code})]: {e}")
            continue

    print(f"\n✅ 다운로드 완료! 성공: {success}개")
    print(f"📁 저장 위치: {SAVE_DIR}")

if __name__ == "__main__":
    download_kr_data()

📋 한국 종목 리스트 수집 및 병합 중...
   - KOSPI 리스트 가져오는 중...
   - KOSDAQ 리스트 가져오는 중...
   - 한국 ETF 리스트 가져오는 중...
✅ 리스트 병합 완료: 총 3839 -> 3839개 (중복 제거됨)
🚀 총 3839개 한국 종목 다운로드 시작...


  0%|          | 2/3839 [00:20<05:47, 11.04it/s]